# GraphKV：相关子图读取＋缓存一致性保护＋普通顺序备用路径

这是研究原型，不覆盖 v5 文件。**新内核，从上到下运行**；Notebook 所在目录应包含 `graphkv_safe/` 与 `tests/`。

本实验回答三个问题：① 固定完整缓存，只改变可见范围，还会不会从银翻转为铜？② 实际按问题选节点后，图模式与读取同样证据的普通顺序模式各自怎样？③ 缓存及其依赖被改变后，能否及时拒绝使用旧状态？

目前选择器是明确实体名匹配，不是通用检索系统。备用路径只表示改用普通顺序读取，不能自动保证答案正确。自动评分仅核对材料集合，完整关系与引用仍需人工检查。这里的时间是运行记录，**不能作为公平的缓存加速成绩**。


In [1]:
from pathlib import Path
import sys, os, json, hashlib, datetime, subprocess, time, gc

PACKAGE_DIR = Path.cwd()  # 解压后，从此目录打开 notebook
REPO_DIR = Path('/home/lyh/hcy_work/GraphKV_refactor_v5')
MODEL_PATH = Path('/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT')
PHYSICAL_GPU = '3'
MAX_NEW_TOKENS = 64
CONTEXT_BUDGET = 4096
RUN_SYNTHETIC = True
SYNTHETIC_SPLIT = 'pilot'  # pilot 看流程；策略定好后再用 dev / test
N_SYNTHETIC_GRAPHS = 8   # 默认 8 图×3 问×4 路径；不是充分的泛化评估
RUN_CPU_FP32 = True     # 有充足 CPU 内存时设 True；只复核原小图
CPU_THREADS = 8

assert (PACKAGE_DIR / 'graphkv_safe' / 'runtime.py').is_file(), '请从完整实验包目录打开 notebook'
if not (REPO_DIR / 'graph_kv_cache.py').is_file():
    REPO_DIR = REPO_DIR / 'GraphKV_refactor_v5'
assert (REPO_DIR / 'graph_kv_cache.py').is_file(), '检查 REPO_DIR'
assert MODEL_PATH.is_dir(), '检查 MODEL_PATH'
if 'torch' in sys.modules:
    raise RuntimeError('请重启内核，从本单元开始运行，以正确设置 GPU')
os.environ['CUDA_VISIBLE_DEVICES'] = PHYSICAL_GPU
sys.path[:0] = [str(PACKAGE_DIR), str(REPO_DIR)]
RUN_DIR = PACKAGE_DIR / 'results' / datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
RESULT_PATH = RUN_DIR / 'graphkv_query_safe_results.json'
RESULTS = dict(schema_version=1, status='in_progress', runs=[], snapshots=[], checks={},
               configuration=dict(max_new_tokens=MAX_NEW_TOKENS, context_budget=CONTEXT_BUDGET,
                                  physical_gpu=PHYSICAL_GPU, synthetic_split=SYNTHETIC_SPLIT,
                                  synthetic_graphs=N_SYNTHETIC_GRAPHS, run_cpu_fp32=RUN_CPU_FP32))
def save():
    temporary = RESULT_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(RESULT_PATH)
print('结果目录：', RUN_DIR)


结果目录： /home/lyh/hcy_work/GraphKV_refactor_v5/GraphKV_query_safe_research/results/20261007_171924_594988


In [2]:
# 核对被研究的 v5 版本，先运行小模型集成测试；不下载任何模型。
EXPECTED_V5 = {'graph_kv_cache.py': '8c6d4796e716573e065204c95689d90104382b2b2d6a6af5f94fa9d13470e452', 'graph_kv_adapter.py': '3c07316e200849882789c1f9310f7cd5c9e760a859c114bb0858df1893256d5f'}
actual = {name: hashlib.sha256((REPO_DIR / name).read_bytes()).hexdigest() for name in EXPECTED_V5}
RESULTS['v5_hashes'] = actual
RESULTS['prototype_hashes'] = {str(p.relative_to(PACKAGE_DIR)): hashlib.sha256(p.read_bytes()).hexdigest()
                                for p in sorted((PACKAGE_DIR / 'graphkv_safe').glob('*.py'))}
save()
assert actual == EXPECTED_V5, 'v5 文件与已分析版本不同：请先核对差异，不要直接忽略此检查'
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join([str(PACKAGE_DIR), str(REPO_DIR), env.get('PYTHONPATH', '')])
env['PYTHONIOENCODING'] = 'utf-8'
test = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                      cwd=PACKAGE_DIR, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test.stdout)
(RUN_DIR / 'integration_tests.txt').write_text(test.stdout, encoding='utf-8')
RESULTS['checks']['integration_test_exit_code'] = test.returncode
RESULTS['checks']['integration_test_log'] = test.stdout
save()
assert test.returncode == 0 and 'skipped' not in test.stdout.lower(), '集成测试失败或被跳过，先停止模型实验'


test_atomic_malformed_update (test_policy_guard.GuardTests) ... ok
test_encoded_dependency_cannot_change (test_policy_guard.GuardTests) ... ok
test_new_target_edges_allowed_before_seal (test_policy_guard.GuardTests) ... ok
test_reassignment_and_layout_changes_detected (test_policy_guard.GuardTests) ... ok
test_seal_blocks_mutating_methods (test_policy_guard.GuardTests) ... ok
test_tensor_inplace_change_detected (test_policy_guard.GuardTests) ... ok
test_tensor_replacement_detected (test_policy_guard.GuardTests) ... ok
test_unsealed_query_rejected (test_policy_guard.GuardTests) ... ok
test_ambiguous_alias (test_policy_guard.PolicyTests) ... ok
test_bad_graphs (test_policy_guard.PolicyTests) ... ok
test_broad_questions_fall_back (test_policy_guard.PolicyTests) ... ok
test_direct_source (test_policy_guard.PolicyTests) ... ok
test_empty_question (test_policy_guard.PolicyTests) ... ok
test_lyra_closure (test_policy_guard.PolicyTests) ... ok
test_malformed_edge_and_alias_containers (test_pol

In [3]:
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from dataclasses import replace
from graphkv_safe.runtime import QueryEngine, cache_digest
from graphkv_safe.guard import FrozenGraphError, StaleCacheError
from graphkv_safe.fixtures import original_case, synthetic_case, serialize_spec, material_screen

assert transformers.__version__ == '4.50.0'
assert torch.cuda.is_available()
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), local_files_only=True,
    torch_dtype=torch.bfloat16, attn_implementation='eager', device_map={'': 0}).eval()
RESULTS['environment'] = dict(torch=torch.__version__, transformers=transformers.__version__,
    cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), model_path=str(MODEL_PATH),
    python=sys.version, model_config=model.config.to_dict(), generation_config=model.generation_config.to_dict())
save()
print('模型已加载：', RESULTS['environment']['gpu'])


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]


模型已加载： NVIDIA GeForce RTX 4090


## 1．建立静态快照，检查保护是否生效

一个快照建好后，不允许改节点或改边。要更新材料、关系、模型权重或位置配置，应新建引擎，旧查询计划会被拒绝。保护失败会直接报错，不能悄悄转用可能已经失效的缓存。


In [4]:
spec, questions = original_case()
torch.cuda.synchronize()
start = time.perf_counter()
engine = QueryEngine(model, tokenizer, spec, model_revision=str(MODEL_PATH), context_budget=CONTEXT_BUDGET)
torch.cuda.synchronize()
RESULTS['snapshots'].append(dict(id='original', spec=serialize_spec(spec), manifest=engine.manifest(),
                                 build_seconds=time.perf_counter() - start))
before = cache_digest(engine.cache)
blocked = {}
try:
    engine.cache.set_edges([('4', '1'), ('4', '2'), ('4', '3'), ('5', '3')])
    blocked['edge_change'] = False
except FrozenGraphError as exc:
    blocked['edge_change'] = True
    print('已拒绝改边：', exc)
try:
    engine.answer_plan(replace(engine.plan(questions[1]['question']), cache_revision='old-cache'))
    blocked['stale_plan'] = False
except StaleCacheError as exc:
    blocked['stale_plan'] = True
    print('已拒绝旧计划：', exc)
blocked['digest_unchanged'] = before == cache_digest(engine.cache)
RESULTS['checks']['lifecycle'] = blocked
save()
assert all(blocked.values())
for q in questions:
    p = engine.plan(q['question'])
    print(q['id'], 'sources=', p.source_ids, 'targets=', p.target_ids, 'reason=', p.reason)


已拒绝改边： The encoded graph is sealed. Build a new snapshot to change it.
已拒绝旧计划： Plan belongs to a different graph/cache snapshot
ORION sources= ('4',) targets= ('1',) reason= explicit_entities_and_graph_dependencies
LYRA sources= ('5',) targets= ('2',) reason= explicit_entities_and_graph_dependencies
NOVA sources= ('4', '5') targets= ('3',) reason= explicit_entities_and_graph_dependencies


## 2．原问题：四种读取方式

- `graph_all`：原全图读取。
- `graph_selected`：自动选出的相关节点，使用原图缓存。
- `sequential_selected`：完全相同的节点和分词结果，普通顺序重新编码。
- `sequential_all`：全部原文，普通顺序重新编码。

不要只比较第一项与第二项；第二项与第三项的差异才有助于判断图缓存还有什么额外价值。`hybrid` 对这三个明确实体问题与 `graph_selected` 走同一路径，另在第 4 节验证备用路径。


In [5]:
MODES = ('graph_all', 'graph_selected', 'sequential_selected', 'sequential_all')
def record(row, *, section, graph_id, question_info=None, mode=None):
    row.update(section=section, graph_id=graph_id, comparison_mode=mode)
    if question_info is not None:
        row['gold_for_scoring_only'] = question_info
        chosen = set(row['plan']['source_ids'] + row['plan']['target_ids'])
        row['evidence_recall'] = len(chosen & set(question_info['evidence'])) / len(question_info['evidence'])
        row['score'] = material_screen(row.get('answer'), question_info['materials']) if row['status'] == 'ok' else None
    RESULTS['runs'].append(row)
    save()
    print(section, graph_id, mode, repr(row.get('answer')), 'status=', row['status'])

for q in questions:
    for mode in MODES:
        record(engine.answer(q['question'], mode=mode, max_new_tokens=MAX_NEW_TOKENS),
               section='original_comparison', graph_id='original', question_info=q, mode=mode)
RESULTS['checks']['original_cache_unchanged'] = before == cache_digest(engine.cache)
save()
assert RESULTS['checks']['original_cache_unchanged']


original_comparison original graph_all 'Copper.' status= ok
original_comparison original graph_selected 'Copper.' status= ok
original_comparison original sequential_selected 'Copper.' status= ok
original_comparison original sequential_all 'Copper' status= ok
original_comparison original graph_all 'Copper.' status= ok
original_comparison original graph_selected 'Silver.' status= ok
original_comparison original sequential_selected 'Silver.' status= ok
original_comparison original sequential_all 'According to Document 2, Project LYRA uses the material supplied by BETA, which is silver.' status= ok
original_comparison original graph_all 'According to the provided documents, Project NOVA combines copper supplied by ALPHA and silver supplied by BETA.' status= ok
original_comparison original graph_selected 'According to the documents, Project NOVA combines copper supplied by ALPHA and silver supplied by BETA.' status= ok
original_comparison original sequential_selected 'According to the docum

## 3．固定缓存长度，单独改变读取范围

四组都保留完整五节点缓存。屏蔽覆盖整个问题及每一个生成步骤。

| 条件 | 可直接读取的节点 |
|---|---|
| A | 全部 |
| B | source 5＋target 2 |
| C | source 4＋source 5＋target 2 |
| D | 除 source 4 以外的全部 |

若 B 答银而 C 答铜，说明这个翻转不需要通过改变完整缓存形状才出现。D 中的 target 1/3 已经读过 source 4，所以它不等于完全移除铜的信息。


In [6]:
conditions = dict(A_all=('4','5','1','2','3'), B_related=('5','2'),
                  C_add_source4=('4','5','2'), D_hide_source4=('5','1','2','3'))
for label, visible in conditions.items():
    record(engine.answer_masked(questions[1]['question'], visible, max_new_tokens=MAX_NEW_TOKENS),
           section='fixed_shape_bf16', graph_id='original', mode=label)
masked = [r for r in RESULTS['runs'] if r['section'] == 'fixed_shape_bf16']
RESULTS['checks']['mask_same_prompt_length'] = len({r['physical_prompt_tokens'] for r in masked}) == 1
RESULTS['checks']['mask_same_logical_start'] = len({r['query_logical_start'] for r in masked}) == 1
baseline = next(r for r in RESULTS['runs'] if r['section'] == 'original_comparison'
                and r['comparison_mode'] == 'graph_all' and r['gold_for_scoring_only']['id'] == 'LYRA')
RESULTS['checks']['all_visible_matches_v5_tokens_bf16'] = masked[0]['token_ids'] == baseline['token_ids']
RESULTS['checks']['mask_cache_unchanged'] = before == cache_digest(engine.cache)
save()
print(json.dumps(RESULTS['checks'], ensure_ascii=False, indent=2))
assert RESULTS['checks']['mask_same_prompt_length'] and RESULTS['checks']['mask_same_logical_start']
assert RESULTS['checks']['mask_cache_unchanged']
if not RESULTS['checks']['all_visible_matches_v5_tokens_bf16']:
    print('注意：全可见 mask 与原适配器 BF16 输出不一致。先看 FP32 复核，暂不作机制结论。')


fixed_shape_bf16 original A_all 'Copper.' status= ok
fixed_shape_bf16 original B_related 'Silver.' status= ok
fixed_shape_bf16 original C_add_source4 'Copper.' status= ok
fixed_shape_bf16 original D_hide_source4 'Silver.' status= ok
{
  "integration_test_exit_code": 0,
  "integration_test_log": "test_atomic_malformed_update (test_policy_guard.GuardTests) ... ok\ntest_encoded_dependency_cannot_change (test_policy_guard.GuardTests) ... ok\ntest_new_target_edges_allowed_before_seal (test_policy_guard.GuardTests) ... ok\ntest_reassignment_and_layout_changes_detected (test_policy_guard.GuardTests) ... ok\ntest_seal_blocks_mutating_methods (test_policy_guard.GuardTests) ... ok\ntest_tensor_inplace_change_detected (test_policy_guard.GuardTests) ... ok\ntest_tensor_replacement_detected (test_policy_guard.GuardTests) ... ok\ntest_unsealed_query_rejected (test_policy_guard.GuardTests) ... ok\ntest_ambiguous_alias (test_policy_guard.PolicyTests) ... ok\ntest_bad_graphs (test_policy_guard.PolicyTe

## 4．备用路径与预算不足

无明确实体、问题范围较广或别名有歧义时，规划器使用完整候选集走普通顺序路径。当前原型没有可靠的“图答案是否正确”检测器，因此不会根据模型自信程度重试。未知实体问题会保存完整答案供人工检查；切换路径不等于模型一定会承认证据不足。

预算不足时返回 `budget_exceeded`，不截掉证据、不启动模型生成。首版没有把故障伪装成成功的备用答案。


In [7]:
for q in ('Which materials are used by all projects?',
          'What material does UNKNOWN_PROJECT use? Answer with the material name only.'):
    record(engine.answer(q, mode='hybrid', max_new_tokens=MAX_NEW_TOKENS),
           section='fallback', graph_id='original', mode='hybrid')
over_budget = engine.answer(questions[1]['question'], max_new_tokens=CONTEXT_BUDGET + 1)
RESULTS['checks']['over_budget'] = over_budget
save()
assert over_budget['status'] == 'budget_exceeded' and over_budget['answer'] is None


fallback original hybrid 'According to the provided documents, no materials are used by all projects. Project ORION uses material supplied by ALPHA (copper, code AX17), Project LYRA uses material supplied by BETA (silver, code BY29), and Project NOVA combines materials supplied by ALPHA and BETA.' status= ok
fallback original hybrid 'UNKNOWN_PROJECT does not use a specific material as it is not mentioned in the provided documents.' status= ok


## 5．独立小图：防止只修好 LYRA

默认 8 个 pilot 图；每个图随机材料配对、名称、文档编号、节点顺序与单项目支持来源，含单来源、多来源、直接事实三类问题。正确答案和必要节点标签只给评分函数，选择器只读取问题、公开别名和图关系。

这仍是同一类合成模板，不能代表真实 RAG。确认流程正常后，可用 `SYNTHETIC_SPLIT='dev'` 跑开发集；规则固定后再用 `'test'` 跑保留集。建议后续分别 30/100 图，并增加缺边、错边、隐含实体、长文档和更多干扰来源。不要反复查看 test 后调规则。


In [8]:
if RUN_SYNTHETIC:
    for index in range(N_SYNTHETIC_GRAPHS):
        generated_spec, generated_questions = synthetic_case(index, split=SYNTHETIC_SPLIT)
        graph_id = f'{SYNTHETIC_SPLIT}-{index:04d}'
        torch.cuda.synchronize()
        start = time.perf_counter()
        candidate = QueryEngine(model, tokenizer, generated_spec, model_revision=str(MODEL_PATH),
                                context_budget=CONTEXT_BUDGET)
        torch.cuda.synchronize()
        RESULTS['snapshots'].append(dict(id=graph_id, spec=serialize_spec(generated_spec),
            manifest=candidate.manifest(), build_seconds=time.perf_counter() - start))
        digest_before = cache_digest(candidate.cache)
        for q in generated_questions:
            for mode in MODES:
                record(candidate.answer(q['question'], mode=mode, max_new_tokens=MAX_NEW_TOKENS),
                       section='synthetic', graph_id=graph_id, question_info=q, mode=mode)
        unchanged = digest_before == cache_digest(candidate.cache)
        RESULTS['checks'][f'cache_unchanged_{graph_id}'] = unchanged
        save()
        assert unchanged
        del candidate
else:
    print('本次未运行独立小图；不能据此声称泛化改善。')


synthetic pilot-0000 graph_all 'Copper' status= ok
synthetic pilot-0000 graph_selected 'Copper.' status= ok
synthetic pilot-0000 sequential_selected 'Copper.' status= ok
synthetic pilot-0000 sequential_all 'copper' status= ok
synthetic pilot-0000 graph_all 'According to the provided documents, Project PRJLQJGME combines aluminum supplied by SUPWVQJXZ and copper supplied by SUPYELNDW.' status= ok
synthetic pilot-0000 graph_selected 'According to the provided documents, Project PRJLQJGME combines aluminum and copper.' status= ok
synthetic pilot-0000 sequential_selected 'According to the documents, Project PRJLQJGME combines aluminum and copper.' status= ok
synthetic pilot-0000 sequential_all 'According to the documents, Project PRJLQJGME combines aluminum (supplied by SUPYELNDW) and copper (supplied by SUPWVQJXZ).' status= ok
synthetic pilot-0000 graph_all 'According to the provided documents, SUPWVQJXZ supplies copper.' status= ok
synthetic pilot-0000 graph_selected 'Copper.' status= ok

## 6．可选 CPU FP32 复核

约需 32 GB 以上权重内存及额外运行内存，耗时取决于 CPU。仅跑原小图全可见基线及四种固定形状条件。该部分可以帮助区分 BF16 的数值因素，但仍不能定位唯一内部原因。


In [9]:
if RUN_CPU_FP32:
    torch.set_num_threads(CPU_THREADS)
    cpu_model = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), local_files_only=True,
        torch_dtype=torch.float32, attn_implementation='eager', device_map={'': 'cpu'}).eval()
    cpu_engine = QueryEngine(cpu_model, tokenizer, spec, model_revision=str(MODEL_PATH), context_budget=CONTEXT_BUDGET)
    RESULTS['snapshots'].append(dict(id='original_cpu_fp32', spec=serialize_spec(spec), manifest=cpu_engine.manifest()))
    cpu_before = cache_digest(cpu_engine.cache)
    cpu_baseline = cpu_engine.answer(questions[1]['question'], mode='graph_all', max_new_tokens=MAX_NEW_TOKENS)
    record(cpu_baseline, section='cpu_reference', graph_id='original_cpu_fp32', mode='graph_all')
    cpu_rows = []
    for label, visible in conditions.items():
        row = cpu_engine.answer_masked(questions[1]['question'], visible, max_new_tokens=MAX_NEW_TOKENS)
        cpu_rows.append(row)
        record(row, section='fixed_shape_fp32', graph_id='original_cpu_fp32', mode=label)
    RESULTS['checks']['all_visible_matches_v5_tokens_fp32'] = cpu_rows[0]['token_ids'] == cpu_baseline['token_ids']
    RESULTS['checks']['cpu_cache_unchanged'] = cpu_before == cache_digest(cpu_engine.cache)
    save()
    assert RESULTS['checks']['cpu_cache_unchanged']
    del cpu_engine, cpu_model
    gc.collect()
else:
    RESULTS['checks']['cpu_fp32_status'] = 'not_run'
    save()


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.33it/s]


cpu_reference original_cpu_fp32 graph_all 'Copper.' status= ok
fixed_shape_fp32 original_cpu_fp32 A_all 'Copper.' status= ok
fixed_shape_fp32 original_cpu_fp32 B_related 'Silver.' status= ok
fixed_shape_fp32 original_cpu_fp32 C_add_source4 'Copper.' status= ok
fixed_shape_fp32 original_cpu_fp32 D_hide_source4 'Silver.' status= ok


## 7．汇总与下一步判断

先检查缓存保护、集成测试与全可见对齐，再读材料评分。**材料集合对了，不代表供应商配对、解释或文档引用对了。** 按基础图查看结果；后续置信区间也应按图分组。

- 子图两条路径都改善：证据筛选有帮助，尚不能归功于图缓存。
- 子图图模式仍差、同文本顺序模式较好：优先顺序备用或局部重算，再考虑训练。
- 子图图模式质量接近且重复查询成本更低：才有继续优化缓存的依据。
- 当前时间不包括公平的顺序前缀缓存基线，不能直接宣布加速。

请保留 `graphkv_query_safe_results.json` 和已运行 notebook。JSON 每条回答后保存，失败时已完成结果仍在。


In [10]:
summary = []
for section in ('original_comparison', 'synthetic'):
    for mode in MODES:
        rows = [r for r in RESULTS['runs'] if r['section'] == section and r['comparison_mode'] == mode]
        if not rows:
            continue
        ok = [r for r in rows if r['status'] == 'ok']
        item = dict(section=section, mode=mode, attempted=len(rows), completed=len(ok),
                    material_set_matches=sum(r['score']['material_set_match'] for r in ok),
                    evidence_complete=sum(r['evidence_recall'] == 1 for r in ok),
                    hit_output_limit=sum(r['hit_limit'] for r in ok),
                    relation_citation_score='requires_human_review')
        summary.append(item)
        print(item)
RESULTS['summary'] = summary
RESULTS['status'] = 'completed'
RESULTS['checks']['final_original_cache_unchanged'] = before == cache_digest(engine.cache)
save()
assert RESULTS['checks']['final_original_cache_unchanged']
print('结果已保存：', RESULT_PATH)
print('尚未验证：真实检索泛化、语义错误自动回退、公平性能基准、多 GPU 模型切分。')


{'section': 'original_comparison', 'mode': 'graph_all', 'attempted': 3, 'completed': 3, 'material_set_matches': 2, 'evidence_complete': 3, 'hit_output_limit': 0, 'relation_citation_score': 'requires_human_review'}
{'section': 'original_comparison', 'mode': 'graph_selected', 'attempted': 3, 'completed': 3, 'material_set_matches': 3, 'evidence_complete': 3, 'hit_output_limit': 0, 'relation_citation_score': 'requires_human_review'}
{'section': 'original_comparison', 'mode': 'sequential_selected', 'attempted': 3, 'completed': 3, 'material_set_matches': 3, 'evidence_complete': 3, 'hit_output_limit': 0, 'relation_citation_score': 'requires_human_review'}
{'section': 'original_comparison', 'mode': 'sequential_all', 'attempted': 3, 'completed': 3, 'material_set_matches': 3, 'evidence_complete': 3, 'hit_output_limit': 0, 'relation_citation_score': 'requires_human_review'}
{'section': 'synthetic', 'mode': 'graph_all', 'attempted': 24, 'completed': 24, 'material_set_matches': 13, 'evidence_comple